In [ ]:
# install dependencies
!pip install ultralytics roboflow -q

In [ ]:
# download skin lesion detection dataset from Roboflow
from roboflow import Roboflow

API_KEY = "4fXsNUvgmczRkVLRm5TI"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("fredrick-gk-wpenz").project("skin-disease-hwlrn")
dataset = project.version(1).download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# inspect class names and structure from the downloaded data.yaml
import yaml
from pathlib import Path

data_yaml_path = Path(dataset.location) / "data.yaml"

with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("Original classes:")
for i, name in enumerate(data_config["names"]):
    print(f"  {i}: {name}")
print(f"\nTotal classes: {data_config['nc']}")

In [ ]:
# remap all class IDs to 0 as this YOLO model will only be used for detection (bounding box), not classification,
# so I'm collapsing the original 6 disease-specific classes into a single "skin_lesion" class
from pathlib import Path

def merge_to_single_class(label_dir):
    """Overwrite all class IDs in YOLO label files to 0 (single class)."""
    label_dir = Path(label_dir)
    if not label_dir.exists():
        print(f"  Skipping {label_dir} (does not exist)")
        return 0

    files = list(label_dir.glob("*.txt"))
    for label_file in files:
        lines = label_file.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            if line.strip() == "":
                continue
            parts = line.split()
            parts[0] = "0"  # overwrite class ID to 0
            new_lines.append(" ".join(parts))
        label_file.write_text("\n".join(new_lines))

    return len(files)

dataset_path = Path(dataset.location)

for split in ["train", "valid", "test"]:
    label_dir = dataset_path / split / "labels"
    count = merge_to_single_class(label_dir)
    print(f"  {split}: remapped {count} label files")

print("\nAll classes merged to class 0 (skin_lesion)")

In [ ]:
# update data.yaml to reflect the single-class setup before training
import yaml
from pathlib import Path

data_yaml_path = Path(dataset.location) / "data.yaml"

with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

# Update to single class
data_config["nc"] = 1
data_config["names"] = ["skin_lesion"]

with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Updated data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

In [ ]:
# verify a sample label file looks correct after remapping
import random

train_labels = list((Path(dataset.location) / "train" / "labels").glob("*.txt"))
sample = random.choice(train_labels)

print(f"Sample label file: {sample.name}")
print(sample.read_text())

In [ ]:
from ultralytics import YOLO

# yolov8s = small model, good balance of speed vs accuracy
model = YOLO("yolov8s.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    name="lesion_detector",
    project="runs/train",
    patience=10,    # early stopping if no improvement after 10 epochs
    save=True,
    exist_ok=True,
    lr0=0.001
)

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/skin-disease-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=lesion_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

In [ ]:
# validate the best checkpoint and print detection metrics
best_model = YOLO("/content/runs/detect/runs/train/lesion_detector/weights/best.pt")
metrics = best_model.val()

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# visualise predictions on a random sample of test images
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import random

import glob
matches = glob.glob("/content/**/samples/*.jpg", recursive=True)
print(matches)

# run predictions on a few test images
test_images = list((Path(dataset.location) / "test" / "images").glob("*.jpg"))
if not test_images:
    test_images = list((Path(dataset.location) / "valid" / "images").glob("*.jpg"))

sample_images = random.sample(test_images, min(6, len(test_images)))

results = best_model.predict(sample_images, conf=0.25, save=True, project="runs/predict", name="samples")

pred_dir = Path(matches[0]).parent  # use the actual directory
pred_images = list(pred_dir.glob("*.jpg"))
print(f"Found {len(pred_images)} images in {pred_dir}")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), pred_images[:6]):
    img = mpimg.imread(str(img_path))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(img_path.name)

plt.tight_layout()
plt.show()

In [ ]:
# confirm the path to best weights
best_weights = Path("/content/runs/train/lesion_detector/weights/best.pt")
print(f"Best model weights saved at:\n{best_weights}")

In [ ]:
# save best weights for use in the pipeline notebook
from google.colab import drive
drive.mount('/drive')

import shutil
shutil.copy(
    "/content/runs/train/lesion_detector/weights/best.pt",
    "/drive/MyDrive/Colab Notebooks/fyp_notebooks/model_weights/yolov8.pt"
)
print("Saved to Google Drive")